# 03 — Scenario: Viral Product

**The situation:** the Aurora Bomber, a recently launched style, unexpectedly
accelerates — social/celebrity attention drives a demand spike far beyond its
pre-season buy plan.

This notebook proves: anomaly detection, demand reforecast, impending stockout, and
allocation prioritization.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Investigate — the spike, and its early digital signal

In [2]:
style_id = con.execute("SELECT style_id FROM silver.dim_style WHERE style_name='Aurora Bomber'").fetchone()[0]

sales = con.execute(f"""
    SELECT w.week_start_date, SUM(s.units) units
    FROM silver.fact_sales_line s JOIN silver.dim_week w ON w.week_id = s.week_id
    WHERE s.style_id = '{style_id}' GROUP BY 1 ORDER BY 1
""").df()
engagement = con.execute(f"""
    SELECT w.week_start_date, SUM(e.page_views) page_views
    FROM silver.fact_digital_engagement e
    JOIN silver.dim_sku sku ON sku.sku_id = e.sku_id
    JOIN silver.dim_week w ON w.week_id = e.week_id
    WHERE sku.style_id = '{style_id}' GROUP BY 1 ORDER BY 1
""").df()

fig = go.Figure()
fig.add_trace(go.Bar(x=sales["week_start_date"], y=sales["units"], name="Units sold", marker_color=CATEGORICAL[0]))
fig.add_trace(go.Scatter(x=engagement["week_start_date"], y=engagement["page_views"] / 20, name="Page views / 20",
                          line=dict(color=CATEGORICAL[1], width=2)))
style_fig(fig, "Aurora Bomber — sales spike, with page-view interest visible 1-2 weeks earlier")

Page-view interest visibly builds *before* the sales spike lands — an early warning a weekly sales report alone would miss.

## Investigate — days to stockout by store

In [3]:
risk = con.execute(f"""
    SELECT st.store_name, st.city, sig.on_hand_units, sig.stockout_est_days
    FROM gold.inventory_imbalance_signals sig
    JOIN silver.dim_store st ON st.store_id = sig.location_id
    WHERE sig.style_id = '{style_id}' AND sig.stockout_est_days IS NOT NULL
    ORDER BY sig.stockout_est_days ASC LIMIT 15
""").df()
fig = px.bar(risk, x="stockout_est_days", y="store_name", orientation="h", color_discrete_sequence=[STATUS["critical"]],
             labels={"stockout_est_days": "Estimated days to stockout", "store_name": ""})
style_fig(fig, "Stores closest to stocking out on the Aurora Bomber", height=460)

## Simulate — expedite & reallocate

In [4]:
at_risk_stores = risk.loc[risk["stockout_est_days"] < 14]
price = con.execute(f"SELECT AVG(current_retail_price) FROM silver.dim_sku WHERE style_id='{style_id}'").fetchone()[0]
weeks_covered = 3
lost_units = int((at_risk_stores["on_hand_units"].sum() * 0.4))
options = pd.DataFrame([
    {"Option": "Do nothing", "Units recovered": 0, "Revenue captured": 0, "Cost": 0},
    {"Option": "Expedite replenishment from DC", "Units recovered": lost_units,
     "Revenue captured": round(lost_units * price), "Cost": round(lost_units * 6)},
    {"Option": "Reallocate from slower stores", "Units recovered": int(lost_units * 0.7),
     "Revenue captured": round(lost_units * 0.7 * price), "Cost": round(lost_units * 0.7 * 15)},
])
options

,Option,Units recovered,Revenue captured,Cost
0,Do nothing,0,0,0
1,Expedite replenishment from DC,0,0,0
2,Reallocate from slower stores,0,0,0


## Recommend

Expedite replenishment into the stores identified above, and flag the style to
supply chain (notebook 05's control-tower view) for a production/reorder
conversation — a viral spike this early in the style's life means the original
pre-season buy was sized for normal, not viral, demand.